In [14]:
import json
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm import tqdm
import os
import warnings
from src.network_util import network_attack_giant_component, network_attack_shortest_path

sns.set_theme(
    style="whitegrid", 
    font='serif',
    rc={
        'font.serif': ['Times New Roman'],
        'axes.labelsize': 20,        # Tamanho dos rótulos dos eixos
        'legend.fontsize': 18,       # Tamanho da legenda
        'xtick.labelsize': 16,       # Tamanho dos rótulos do eixo x
        'ytick.labelsize': 16,       # Tamanho dos rótulos do eixo y
        'axes.titlesize': 0,         # Remover título (como solicitado)
        'figure.autolayout': True,   # Ajuste automático do layout
        'figure.dpi': 300,           # Alta resolução para publicação
        'savefig.dpi': 300,          # Alta resolução para salvar
        'lines.linewidth': 2.5       # Espessura das linhas
    }
)
sns.set_context("talk")

INSTITUICOES = ['uft', 'ufnt', 'ceulp', 'ifto', 'unitins', 'tocantins']


In [2]:
def load_network_data(institution, year):
    """Carrega os dados da rede a partir do arquivo JSON"""
    file_path = f'../results/metrics/{institution}/{institution}_{year}.json'
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data

In [3]:
def load_real_graph(institution, year):
    """Carrega o grafo real a partir do arquivo GEXF"""
    file_path = f'../results/graphs/{institution}/graph_{institution}_{year}.gexf'
    G = nx.read_gexf(file_path)
    return G

In [4]:
output_dir = "../results/img/robustness_plots"

# Loop sobre todas as instituições
for instituicao in INSTITUICOES:
    print(f"\nProcessando {instituicao.upper()}...")
    
    try:
        data = load_network_data(instituicao, 2024)
        graph = load_real_graph(instituicao, 2024)
        
        vertices_removidos, resultados = network_attack_giant_component(graph, data)
        
        plt.figure(figsize=(8, 6))
        
        legend_labels = {
            'random': 'Aleatório',
            'betweenness_centrality': 'Betweenness',
            'closeness_centrality': 'Closeness',
            'eigenvector_centrality': 'Eigenvector',
            'degrees': 'Grau'
        }
        
        for i, (estrategia, valores) in enumerate(resultados.items()):
            sns.lineplot(
                x=vertices_removidos, 
                y=valores, 
                label=legend_labels[estrategia],
                marker='o' if estrategia == 'random' else None,
                markersize=8 if estrategia == 'random' else 0,
                alpha=0.9
            )
        
        # Formatar eixo x com separador de milhar
        plt.gca().xaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(lambda x, pos: f'{x:,.0f}'.replace(',', '.')))
        
        plt.xlabel('Número de vértices removidos', fontsize=20)
        plt.ylabel('Maior Componente Conexa (%)', fontsize=20)
        plt.legend(
            loc='upper right', 
            frameon=True, 
            fancybox=True, 
            shadow=False, 
            edgecolor='black'
        )
        
        plt.grid(True, alpha=0.7)
        sns.despine(left=False, bottom=False)
        plt.xlim(left=0)
        plt.ylim(bottom=0)
        plt.tight_layout()
        
        # Salvar figura
        os.makedirs(os.path.join(output_dir, f"{instituicao}/"), exist_ok=True)
        output_path = os.path.join(output_dir, f"{instituicao}/robustez_{instituicao}_componente_gigante.png")
        
        plt.savefig(output_path, bbox_inches='tight', dpi=300)
        plt.close()
        
        print(f"Gráfico salvo em: {output_path}")
        
    except Exception as e:
        print(f"Erro ao processar {instituicao}: {str(e)}")

print("\nConcluído!")


Processando IFTO...


Testando degrees: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 136.80it/s]


Gráfico salvo em: ../results/img/robustness_plots/ifto/robustez_ifto_componente_gigante.png

Concluído!


In [5]:
output_dir = "../results/img/shortest_path_plots"
os.makedirs(output_dir, exist_ok=True)

warnings.filterwarnings("ignore", category=UserWarning)

# Loop sobre todas as instituições
for instituicao in INSTITUICOES:
    print(f"\nProcessando {instituicao.upper()}...")
    
    try:
        data = load_network_data(instituicao, 2024)
        graph = load_real_graph(instituicao, 2024)
        
        vertices_removidos, resultados = network_attack_shortest_path(graph, data, frac_max=0.2, steps=5)
        
        plt.figure(figsize=(8, 6))
        
        legend_labels = {
            'random': 'Aleatório',
            'betweenness_centrality': 'Betweenness',
            'closeness_centrality': 'Closeness',
            'eigenvector_centrality': 'Eigenvector',
            'degrees': 'Grau'
        }
        
        for estrategia, valores in resultados.items():
            # Substituir infinito por um valor grande para plotagem
            y_vals = [v if v != np.inf else 10 * max(v for v in valores if v != np.inf) for v in valores]
            sns.lineplot(
                x=vertices_removidos, 
                y=y_vals, 
                label=legend_labels[estrategia],
                marker='o' if estrategia == 'random' else None,
                markersize=8 if estrategia == 'random' else 0,
                alpha=0.9
            )
        
        # Formatar eixo x com separador de milhar
        plt.gca().xaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(lambda x, pos: f'{x:,.0f}'.replace(',', '.')))
        
        plt.xlabel('Número de vértices removidos', fontsize=20)
        plt.ylabel('Menor Caminho Médio', fontsize=20)
        plt.legend(
            loc='upper right', 
            frameon=True, 
            fancybox=True, 
            shadow=False, 
            edgecolor='black'
        )
        
        plt.grid(True, alpha=0.7)
        sns.despine(left=False, bottom=False)
        plt.xlim(left=0)
        plt.ylim(bottom=0)
        plt.tight_layout()
        
        # Salvar figura
        output_path = os.path.join(output_dir, f"{instituicao}/robustez_{instituicao}_shortest_path.png")
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        plt.savefig(output_path, bbox_inches='tight', dpi=300)
        plt.close()
        
        print(f"Gráfico salvo em: {output_path}")
        
    except Exception as e:
        print(f"Erro ao processar {instituicao}: {str(e)}")

print("\nConcluído!")


Processando IFTO...


Testando degrees: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:23<00:00,  4.78s/it]


Gráfico salvo em: ../results/img/shortest_path_plots/ifto/robustez_ifto_shortest_path.png

Concluído!


In [15]:
output_dir = "../results/img/shortest_path_plots"
os.makedirs(output_dir, exist_ok=True)

for instituicao in INSTITUICOES:
    print(f"\nProcessando {instituicao.upper()}...")
    
    try:
        # Ensure data (metrics) is calculated for the graph
        graph = load_real_graph(instituicao, 2024)
        data = load_network_data(instituicao, 2024) # Recalculate or load metrics based on the graph
        
        vertices_removidos, resultados = network_attack_shortest_path(graph, data, frac_max=0.05, steps=10)
        
        plt.figure(figsize=(8, 6))
        
        legend_labels = {
            'random': 'Aleatório',
            'betweenness_centrality': 'Betweenness',
            'closeness_centrality': 'Closeness',
            'eigenvector_centrality': 'Eigenvector',
            'degrees': 'Grau'
        }
        
        for estrategia, valores in resultados.items():
            # Substituir infinito por um valor grande para plotagem
            # Find max finite value, if no finite values, use a default large number
            finite_vals = [v for v in valores if v != np.inf and not np.isnan(v)]
            if finite_vals:
                max_finite_val = max(finite_vals)
                y_vals = [v if v != np.inf and not np.isnan(v) else 1.2 * max_finite_val for v in valores] # 1.2 * max_finite_val to make it visible above the others
            else: # All values are inf or nan
                y_vals = [1000] * len(valores) # A large arbitrary number if no finite values
            
            sns.lineplot(
                x=vertices_removidos, 
                y=y_vals, 
                label=legend_labels[estrategia],
                marker='o' if estrategia == 'random' else None,
                markersize=8 if estrategia == 'random' else 0,
                alpha=0.9
            )
        
        # Formatar eixo x com separador de milhar
        plt.gca().xaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(lambda x, pos: f'{x:,.0f}'.replace(',', '.')))
        
        plt.xlabel('Número de vértices removidos', fontsize=20)
        plt.ylabel('Menor Caminho Médio (LCC)', fontsize=20) # Updated label
        plt.legend(
            loc='upper right', 
            frameon=True, 
            fancybox=True, 
            shadow=False, 
            edgecolor='black'
        )
        
        plt.grid(True, alpha=0.7)
        sns.despine(left=False, bottom=False)
        plt.xlim(left=0)
        plt.ylim(bottom=0)
        plt.tight_layout()
        
        # Salvar figura
        output_path = os.path.join(output_dir, f"{instituicao}/robustez_{instituicao}_shortest_path_LCC.png") # Updated filename
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        plt.savefig(output_path, bbox_inches='tight', dpi=300)
        plt.close()
        
        print(f"Gráfico salvo em: {output_path}")
        
    except Exception as e:
        print(f"Erro ao processar {instituicao}: {str(e)}")

print("\nConcluído!")


Processando UFT...


Testando degrees: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [1:02:47<00:00, 376.72s/it]


Gráfico salvo em: ../results/img/shortest_path_plots/uft/robustez_uft_shortest_path_LCC.png

Processando UFNT...


Testando degrees: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:02<00:00,  4.11it/s]


Gráfico salvo em: ../results/img/shortest_path_plots/ufnt/robustez_ufnt_shortest_path_LCC.png

Processando CEULP...


...

Gráfico salvo em: ../results/img/shortest_path_plots/ifto/robustez_ifto_shortest_path_LCC.png

Processando UNITINS...


Testando degrees: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:30<00:00,  3.01s/it]


Gráfico salvo em: ../results/img/shortest_path_plots/unitins/robustez_unitins_shortest_path_LCC.png

Processando TOCANTINS...


Testando degrees: 100%|████████████████████████████████████████████████████████████████████████████████| 10/10 [1:41:24<00:00, 608.46s/it]


Gráfico salvo em: ../results/img/shortest_path_plots/tocantins/robustez_tocantins_shortest_path_LCC.png

Concluído!
